# 09 规则挖掘（report.mining）

基于 `hengshucredit_yyp.xlsx` 或 `hscredit_yyp.xlsx`（`examples/`）进行规则挖掘，输出以中文字段为主，并尽量直接使用 `Rule.report` 的命中结果。

> 挖掘时会自动剔除逾期天数字段（如 `CURRENT_DPD` / 含“逾期”/`DPD` 的字段）。


In [1]:
import os, sys
project_root = os.path.abspath('..').replace('\\', '/')
if project_root not in [p.replace('\\', '/') for p in sys.path]:
    sys.path.append(project_root)


In [2]:
import warnings
warnings.filterwarnings('ignore')

import re
import numpy as np
import pandas as pd

from hscredit import init_setting
from hscredit.report.mining import (
    SingleFeatureRuleMiner,
    MultiFeatureRuleMiner,
    TreeRuleExtractor,
)

init_setting()
pd.set_option('display.max_columns', None)
# 或者设置为一个足够大的数，比如 500
pd.set_option('display.max_columns', 500)


In [3]:
# 读取样例数据（优先 hengshucredit_yyp.xlsx）
from pathlib import Path
_roots = [Path.cwd(), Path.cwd() / 'examples', Path.cwd().parent]
_fp = None
for _r in _roots:
    for _n in ('hengshucredit_yyp.xlsx', 'hscredit_yyp.xlsx'):
        if (_r / _n).is_file():
            _fp = _r / _n
            break
    if _fp is not None:
        break
if _fp is None:
    raise FileNotFoundError('请将 hengshucredit_yyp.xlsx 或 hscredit_yyp.xlsx 放在 examples/')
df = pd.read_excel(_fp)
print(f'数据形状: {df.shape}')
print('字段列表:')
print(df.columns.tolist())


数据形状: (970, 18)
字段列表:
['客户编号', '放款时间', '放款金额', '商品类别', 'MOB1', 'CURRENT_DPD', '中智小牛分C3', '珊瑚92', '极光欺诈分6v1', '青云24', '占信V3', '轻花老客海纳子分V1', '天创小额网贷分', '近六个月非银多头机构数', '手机号近一个月非银多头机构数', '身份证近一个月非银多头机构数', '衡枢鉴真分老客版', 'FPD']


In [4]:
# 自动识别目标变量（优先 FPD）
if 'FPD' in df.columns:
    target = 'FPD'
else:
    target = df.columns[-1]

# 统一目标为二分类 0/1
if not pd.api.types.is_numeric_dtype(df[target]):
    df[target] = pd.to_numeric(df[target], errors='coerce')

df = df[df[target].notna()].copy()
df[target] = (df[target] > 0).astype(int)

print('目标变量:', target)
print('目标分布:')
print(df[target].value_counts(dropna=False))


目标变量: FPD
目标分布:
FPD
0    834
1    136
Name: count, dtype: int64


In [5]:
# 挖掘前剔除逾期天数字段
pattern = re.compile(r'(商品类别|逾期|MOB|dpd|days?|day)', re.IGNORECASE)
overdue_day_cols = [c for c in df.columns if pattern.search(str(c))]

# 也剔除常见ID/时间字段（避免无效规则）
id_time_cols = [c for c in df.columns if any(k in str(c).lower() for k in ['id', '编号', '日期', '时间'])]

exclude_cols = sorted(set(overdue_day_cols + id_time_cols) - {target})

df_mining = df.drop(columns=exclude_cols, errors='ignore').copy()

print('剔除字段:')
print(exclude_cols)
print('建模字段数:', len(df_mining.columns), '（含目标）')


剔除字段:
['CURRENT_DPD', 'MOB1', '商品类别', '客户编号', '放款时间']
建模字段数: 13 （含目标）


## 1) 单特征规则挖掘

In [ ]:
from hscredit import OptimalBinning, feature_bin_stats

In [ ]:
feature_bin_stats(df, feature="中智小牛分C3", method="mdlp", max_n_bins=5, min_bin_size=0.01, monotonic='auto_asc_desc', overdue=["MOB1"], dpds=[3, 0])

In [ ]:
df_mining

,放款金额,中智小牛分C3,珊瑚92,极光欺诈分6v1,青云24,占信V3,轻花老客海纳子分V1,天创小额网贷分,近六个月非银多头机构数,手机号近一个月非银多头机构数,身份证近一个月非银多头机构数,衡枢鉴真分老客版,FPD
0,1399,NaN,NaN,NaN,656,NaN,NaN,630,51,15,15,0.0242,0
1,1399,NaN,NaN,NaN,565,NaN,NaN,583,56,6,18,0.0492,0
2,3960,NaN,NaN,NaN,708,NaN,NaN,764,68,17,20,0.0546,0
3,3960,NaN,NaN,NaN,555,NaN,NaN,712,45,15,15,0.0899,0
4,1399,NaN,NaN,NaN,581,NaN,NaN,641,67,32,32,0.0678,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
965,1399,570.0000,553.0000,0.1192,577,559.0000,0.0655,759,58,19,19,0.1597,0
966,1379,527.0000,603.0000,0.5000,576,551.0000,0.0319,684,47,24,25,0.0381,0
967,3544,624.0000,551.0000,0.2375,683,608.0000,0.0455,769,62,20,20,0.0921,0
968,1379,595.0000,663.0000,0.0110,728,674.0000,0.0353,658,74,25,26,0.1405,0


In [ ]:
single_miner = SingleFeatureRuleMiner(
    target=target,
    method='mdlp',
    max_n_bins=6,
    min_bin_size=0.01,
    min_lift=1.1,
    verbose=False,
)

single_miner.fit(df_mining)
single_top = single_miner.get_top_rules(
    top_n=20,
    metric='lift',
    datasets=df,
    target=target,
)

# show_cols = ['规则表达式', '命中样本数', '命中样本占比', '命中坏样本率', '命中LIFT值', '坏账改善']
single_top #[show_cols]


""


## 2) 双特征交叉规则挖掘

In [ ]:
multi_miner = MultiFeatureRuleMiner(
    target=target,
    method='quantile',
    max_n_bins=5,
    min_samples=5,
    min_lift=1.5,
    verbose=False,
)

multi_miner.fit(df_mining)
multi_top = multi_miner.get_all_cross_rules(
    top_n=5,
    metric='lift',
    max_feature_pairs=20,
    min_samples=30,
    min_lift=1.5,
)

show_cols = ['特征组合', '规则表达式', '命中样本数', '命中样本占比', '命中坏样本率', '命中LIFT值', '坏账改善']
multi_top[show_cols].head(20)


TypeError: could not convert string to float: '礼包'

## 3) 树模型规则提取

In [ ]:
X = df_mining.drop(columns=[target])
y = df_mining[target]

tree_miner = TreeRuleExtractor(
    algorithm='rf',
    target=target,
    n_estimators=20,
    max_depth=4,
    min_samples_leaf=20,
    random_state=42,
)

tree_miner.fit(X, y)
tree_rules_df = tree_miner.get_rules_dataframe(
    top_n=20,
    datasets=df_mining,
    target=target,
    min_samples=30,
    min_confidence=0.0,
)

tree_rules_df.head(20)


## 4) 获取 Rule 对象（可直接用于 Rule.report / ruleset_report）

In [ ]:
single_rules = single_miner.get_rules(top_n=10, datasets=df_mining, target=target)
multi_rules = multi_miner.get_rules(top_n=10, datasets=df_mining, target=target)
tree_rules = tree_miner.get_rules(top_n=10, datasets=df_mining, target=target)

print('单特征规则数:', len(single_rules))
print('双特征规则数:', len(multi_rules))
print('树规则数:', len(tree_rules))

# 示例：查看第一条规则的中文元数据（来自Rule.report命中结果）
if single_rules:
    single_rules[0].metadata_
